# Inspect Consolidated Recharge Datasets

Load and visualize the three recharge source datasets at the annual time-step for year 2000.

Sources (see `catalog/variables.yml` → `recharge`):
- Reitz 2017 (`total_recharge`, inches/year) — already annual
- WaterGAP 2.2d (`qrdif`, kg m-2 s-1) — monthly, summed to annual on the fly
- ERA5-Land (`ssro`, m/month) — monthly, summed to annual on the fly

Note: monthly sources are summed to annual here so issues in any one month surface in the annual cell. The actual recharge target builder does the same aggregation downstream.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import xarray as xr

DATASTORE = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/nhf-datastore")
PROJECT = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets")

TARGET_YEAR = 2000

## Load all three datasets

In [ ]:
datasets = {
    "Reitz 2017 (total_recharge)": {
        "path": DATASTORE / "reitz2017" / "reitz2017_consolidated.nc",
        "var": "total_recharge",
        "time_step": "annual",
        "units": "inches/year",
    },
    "WaterGAP 2.2d (qrdif)": {
        "path": DATASTORE / "watergap22d" / "watergap22d_qrdif_cf.nc",
        "var": "qrdif",
        "time_step": "monthly",
        "units": "kg m-2 s-1 (summed to annual)",
    },
    "ERA5-Land (ssro)": {
        "path": DATASTORE / "era5_land" / f"era5_land_monthly_{TARGET_YEAR}.nc",
        "var": "ssro",
        "time_step": "monthly",
        "units": "m (summed to annual)",
    },
}

opened = {}
for label, info in datasets.items():
    nc_path = info["path"]
    if not nc_path.exists():
        print(f"SKIP {label}: {nc_path} not found (run fetch first)")
        continue
    ds = xr.open_dataset(nc_path)
    opened[label] = (ds, info)
    print(f"{label}: {list(ds.data_vars)} | time: {ds.time.values[0]} .. {ds.time.values[-1]} | shape: {dict(ds.sizes)}")

## Dataset representations

In [ ]:
for label, (ds, _) in opened.items():
    print(f"{'=' * 60}\n{label}\n{'=' * 60}")
    display(ds)

## Plot annual values for target year

In [ ]:
n = len(opened)
if n == 0:
    print("No datasets available yet. Run the fetch commands first.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    axes = axes.flatten() if n > 1 else [axes]

    for idx, (label, (ds, info)) in enumerate(opened.items()):
        ax = axes[idx]
        var = info["var"]
        if info["time_step"] == "monthly":
            da = ds[var].sel(
                time=slice(f"{TARGET_YEAR}-01", f"{TARGET_YEAR}-12")
            ).sum("time")
            label_time = f"{TARGET_YEAR} (sum of 12 months)"
        else:
            da = ds[var].sel(time=str(TARGET_YEAR), method="nearest")
            label_time = str(da.time.values)[:10] if da.time.size else str(TARGET_YEAR)

        da.plot(ax=ax, cmap="BuGn", robust=True)
        ax.set_title(f"{label}\n{label_time} | {info['units']}", fontsize=11)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

    for idx in range(len(opened), len(axes)):
        axes[idx].set_visible(False)

    fig.suptitle(f"Recharge — annual values for {TARGET_YEAR}", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

## Clean up

In [ ]:
for label, (ds, _) in opened.items():
    ds.close()